#**Pandas Text Methods (.str)**

Text data often needs cleaning before analysis: fixing formatting, extracting parts of strings, or validating entries.

Instead of writing custom `apply()` functions, Pandas provides string methods through the `.str` accessor.

These mirror standard Python string methods (like `.lower(), .split(), .strip()`), but work element-wise on a Series or column.


## **Why `.str`?**

- Works on entire columns at once
- Cleaner and more readable than loops or `apply()`
- Familiar if you already know Python strings

**Note:** Older Pandas versions store text as object. Newer versions support a dedicated `string` dtype, but `.str` works the same in both.

## **Basic String Operations**

In [ ]:
import pandas as pd

users = pd.Series(['alice', 'BOB', 'Charlie', '7'])

In [ ]:
users.str.lower()

,0
0,alice
1,bob
2,charlie
3,7


In [ ]:
users.str.upper()

,0
0,ALICE
1,BOB
2,CHARLIE
3,7


In [ ]:
users.str.isdigit()

,0
0,False
1,False
2,False
3,True


This returns a Series of results, which you can directly use for filtering:

In [ ]:
users[~users.str.isdigit()]

,0
0,alice
1,BOB
2,Charlie


## **Splitting, Selecting, and Expanding**

In [ ]:
codes = pd.Series(['TV-SAMSUNG-55', 'PHONE-APPLE-13'])

Split on -:

In [ ]:
codes.str.split('-')

,0
0,"[TV, SAMSUNG, 55]"
1,"[PHONE, APPLE, 13]"


Grab one part:

In [ ]:
codes.str.split('-').str[0]   # category

,0
0,TV
1,PHONE


Expand into columns:

In [ ]:
codes.str.split('-', expand=True)

,0,1,2
0,TV,SAMSUNG,55
1,PHONE,APPLE,13


This turns one column into a DataFrame, which is very common in real datasets.

## **Cleaning Messy Text**

In [ ]:
responses = pd.Series(['  yes ', 'no!', ' YES  '])

Chain multiple string methods:

In [ ]:
responses.str.lower().str.replace('!', '', regex=False).str.strip()

,0
0,yes
1,no
2,yes


`.str.strip()` removes whitespace from both ends of a string.

That includes: spaces, tabs (\t), newline characters (\n)

<br>

`regex=False`: By setting regex=False, you are telling Pandas to treat '!' simply as the literal character exclamation mark. This is important when you want to replace exact strings that might happen to contain characters with special meaning in regular expressions (like ., *, +, ?, |, (, ), [, ], {, }, ^, $, \).

## **`.str` vs `apply()`**

You can always write a custom function:

In [ ]:
def clean_text(x):
    return x.lower().replace('!', '').strip()

responses.apply(clean_text)

,0
0,yes
1,no
2,yes


Which should you use?

- `.str` → simpler, more readable for common tasks

-  `apply()` → needed for complex logic (conditions, loops)

- Performance differences usually matter only for large datasets

Your time as a developer is often more valuable than micro-optimizing small DataFrames.

## **Using `.str` on a DataFrame column**

In [ ]:
import pandas as pd

df = pd.DataFrame({
    'email': ['alice@gmail.com', 'bob@yahoo.com', 'admin@school.edu'],
    'status': [' Active ', 'inactive ', ' ACTIVE']
})
df

,email,status
0,alice@gmail.com,Active
1,bob@yahoo.com,inactive
2,admin@school.edu,ACTIVE


In [ ]:
df['status'] = df['status'].str.strip().str.lower()
df

,email,status
0,alice@gmail.com,active
1,bob@yahoo.com,inactive
2,admin@school.edu,active


In [ ]:
df[['user', 'domain']] = df['email'].str.split('@', expand=True)
df

,email,status,user,domain
0,alice@gmail.com,active,alice,gmail.com
1,bob@yahoo.com,inactive,bob,yahoo.com
2,admin@school.edu,active,admin,school.edu


In [ ]:
df[['status']] = df[['status']].apply(lambda col: col.str.strip())
df

,email,status,user,domain
0,alice@gmail.com,active,alice,gmail.com
1,bob@yahoo.com,inactive,bob,yahoo.com
2,admin@school.edu,active,admin,school.edu


In [ ]:
df['status'] = df['status'].str.strip()
df

,email,status,user,domain
0,alice@gmail.com,active,alice,gmail.com
1,bob@yahoo.com,inactive,bob,yahoo.com
2,admin@school.edu,active,admin,school.edu


<br> <br>

# **Pandas Time Methods (.dt)**

Many datasets include timestamps: logins, transactions, sensor readings, etc.
Python has a built-in datetime object, and Pandas extends it with powerful tools for working with time-based data.

Just like `.str` for text, Pandas provides `.dt` methods to extract information from datetime columns.

These methods are especially useful for:

- Feature engineering
- Aggregations over time
- Preparing data for machine learning models

In [ ]:
from datetime import datetime

dt = datetime(2022, 7, 15, 14, 30)

In [ ]:
dt.year

2022

In [ ]:
dt.month

7

In [ ]:
dt.hour

14

## **Converting Strings to Datetime (pd.to_datetime)**

Real-world data often stores dates as strings.

In [ ]:
import pandas as pd

dates = pd.Series([
    'March 5, 2021',
    '2022-01-01',
    None
])
dates

,0
0,"March 5, 2021"
1,2022-01-01
2,None


In [ ]:
dates1 = pd.to_datetime(dates, errors='coerce')
dates1

,0
0,2021-03-05
1,NaT
2,NaT


In [ ]:
dates2 = pd.to_datetime(dates, format='mixed', errors='coerce')
dates2

,0
0,2021-03-05
1,2022-01-01
2,NaT


- `errors='coerce'` → invalid parsing becomes NaT instead of throwing an error


- Works fine for a mix of `"March 5, 2021"` and `"2022-01-01"`

## **Ambiguous Dates (US vs Europe)**

In [ ]:
pd.to_datetime('10-12-2000')

Timestamp('2000-10-12 00:00:00')

By default, Pandas assumes month-first (US style).

To force day-first:

In [ ]:
pd.to_datetime('10-12-2000', dayfirst=True)

Timestamp('2000-12-10 00:00:00')

⚠️ If your dataset mixes US and European formats, the issue is bad data, not Pandas.

## **Custom Date Formats**

In [ ]:
custom = '15--AUG--2019'
pd.to_datetime(custom, format='%d--%b--%Y')

Timestamp('2019-08-15 00:00:00')

## **Datetime Columns in DataFrames**

In [ ]:
df = pd.DataFrame({
    'timestamp': ['2023-01-01 09:15', '2023-01-02 18:45'],
    'sales': [120, 95]
})

df

,timestamp,sales
0,2023-01-01 09:15,120
1,2023-01-02 18:45,95


In [ ]:
df['timestamp'] = pd.to_datetime(df['timestamp'])
df

,timestamp,sales
0,2023-01-01 09:15:00,120
1,2023-01-02 18:45:00,95


Now Pandas recognizes this column as datetime.

## **Extracting Features with .dt**

In [ ]:
df['hour'] = df['timestamp'].dt.hour
df

,timestamp,sales,hour
0,2023-01-01 09:15:00,120,9
1,2023-01-02 18:45:00,95,18


In [ ]:
df['day_of_week'] = df['timestamp'].dt.day_name()
df

,timestamp,sales,hour,day_of_week
0,2023-01-01 09:15:00,120,9,Sunday
1,2023-01-02 18:45:00,95,18,Monday


In [ ]:
df['is_weekend'] = df['timestamp'].dt.weekday >= 5
df

,timestamp,sales,hour,day_of_week,is_weekend
0,2023-01-01 09:15:00,120,9,Sunday,True
1,2023-01-02 18:45:00,95,18,Monday,False


In [ ]:
pd.__version__

'2.2.2'

<br><br>

### **Accessor**

In Pandas, an "accessor" is a special property that provides access to a specific set of methods and attributes for a particular data type within a Series or DataFrame column. It's essentially a way to group related functionalities.

For example:

*   `.str` accessor: When you have a Series containing string data, the `.str` accessor (`Series.str`) gives you access to a wide range of string manipulation methods (like `.lower()`, `.upper()`, `.strip()`, `.split()`, `.replace()`, etc.). These methods then operate element-wise on each string in the Series.

*  `.dt` accessor: Similarly, for a Series containing datetime objects, the .dt accessor (Series.dt) provides methods and attributes specific to dates and times (like `.year`, `.month`, `.day_name()`, `.hour`, `.weekday`, etc.).

Think of it as a specialized interface that unlocks relevant operations for that data type, making your code cleaner and more efficient than applying functions row by row.

While Pandas `.str` methods are inspired by Python's built-in string methods, there are crucial differences, especially in how they interact with Pandas Series:

* Element-wise Operation:

  Python str methods (e.g., 'hello'.upper()) operate on a single string object.
  Pandas .str methods (e.g., my_series.str.upper()) operate element-wise on every string within a Pandas Series (or DataFrame column). This means you don't need to write explicit loops or use apply() for common string operations across an entire column.

* Handling Missing Values (NaN):

  Python str methods will raise an error if you try to call them on a None or NaN value (e.g., None.upper() will fail).
  Pandas .str methods gracefully handle missing values (represented as NaN or None in a Series) by returning NaN for those elements, without raising an error.

* Return Type:

  Python str methods return a new string (or a list of strings for methods like .split()).
  Pandas .str methods typically return a Pandas Series or DataFrame (e.g., str.split(expand=True)), making it easy to integrate back into your DataFrame operations.

* Performance:

  For operations on an entire Series, Pandas .str methods are generally much more optimized and performant than explicitly looping through a Series and applying Python's str methods, as they are implemented in highly optimized C code under the hood.